# CyPhER UI

## Overview

This is an interactive Jupyter-based UI for exploring network traffic statistics
from GraphBLAS matrices and D4M associative arrays.

---
- **Statistics**
    - Stored in a D4M associative array, taken from a tsv or csv file, e.g. (`stats.tsv`)
    
- **Zone matrices** 
    - Derived from sub-statistics of the form  `sub_<metric>_i_j` 
    - Depending on how your `stats.tsv` was constructed, you may also have it recognize the form `sub-<metric>-i_j`
    
- **Raw traffic matrices** stored as either 
    - GraphBLAS `.grb` files inside per-window tar archives OR 
    - D4M Assoc Arrays inside a `.npy` saved array 

    These matrices should represent the traffic in zones for that particular time window. 
---

## UI Structure

The UI consists of four main layers:

1. **Property–Context Grid**  
   Selects which metric and aggregation to view
   
2. **Contextual Controls**  
   Window selectors and ratio controls appear as needed

3. **Zone Matrix View**  
   Displays aggregated values between zones

4. **Plots**  
   - Spy plots
   - Time series plots

---

## Dependencies
- anaconda/2023b module [`$ module load anaconda/2023b`]
- Python 3.9+
- `ipywidgets`
- `plotly` [`$ pip install --user plotly`]
- `numpy`
- `graphblas` [https://github.com/python-graphblas/python-graphblas ] `$ pip install --user python-graphblas`
- `D4M` [https://github.com/Accla/d4m ]

---

## Notes and Getting Started

- Zone indices in sub-statistics are assumed to be **1-indexed**
- Very large sparse matrices (over 1,000,000 entries) will not be shown in the spy plot. 
- Please load your stats file using the `load_stats` function:
    - specify the filename to be read:
        `filename = "path/to/file/stats_filename.tsv"`
    - Stats files are assumed to have columns of the form `sub_{metric}_i_j`. If instead they are `sub-{metric}-i_j`, set 'delim' to '-'
        - Default is "_"
    - If there is a repeated string in every time window that you would like to get rid of, set 'cleaning_string' to the string to delete
        - `CLEANING_STR = "string to delete"`
        - Default is ".8388608"

- Please also define the directory that the tar files that have the subrange matrices for time windows are located using `dir` 
    - Default is the current directory
    - It's assumed that each tar/npy file is named `<row name of stats file>` without the cleaning string
- Lastly, please name the appropriate number of zones with the `zones` variable. 
    - The zone matrix uses (1,1) at the bottom right corner
    
    
To run: Run all cells once the user defined variables are set correctly. 


Note: When graphing, the int to IPv4 string converter is defaulted to Little-Endian. To change, add `endianess='BE'` for Big Endian in the arguments 
for the `build_ui` function.

---

## Folder Structure 

```
Step5_Viz/ 
|
├── code/
|  |
|  ├── Jupyter_UI.ipynb
|  |
|  ├──Jupyter_UI/
|  |  ├── __init__.py
|  |  ├── build_ui.py
|  |  ├── controls.py
|  |  ├── plotting.py
|  |  ├── sections.py 
|  |  ├── state.py
|  |  ├── theme.py
|  |  ├── utils.py
|
├── data/
|  ├── 20240507-152247.tar 
|  ├── 20240507-152247.npy
|  ├── ... (other time window subrange files)
```


In [1]:
import ipywidgets as widgets
from Jupyter_UI.utils import load_stats, set_zones
from Jupyter_UI.build_ui import build_ui

### User defined Variables

In [2]:
# directory where the subrange matrix .tar/.npy files are located
directory = '../data'

# for given stats file on CAIDA data:
# filename = "GraphChallenge_analyses_subrange_m_23_concat.tsv"
# load_stats(filename, delim='_', cleaning_string='.8388608', dir=directory)

# for random stats file: 
filename = "stats_random.tsv"
load_stats(filename, delim='-', cleaning_string='.8388608.txt_subrange_try', dir=directory)

zones = ["nonroute", "bogon", "assigned", "CAIDA", "other"]

In [3]:
set_zones(zones)
ui_original, _, _ = build_ui()
ui_assoc_array, _, _ = build_ui(assoc_array=True)
ui_little_end,_,_ = build_ui(endianess="BE")

tab = widgets.Tab(children=[ui_original, ui_assoc_array, ui_little_end])
tab.titles = ['From .grb matrices', 'From Assoc Array', 'Big Endian']

display(tab)

In [4]:
# !jupyter nbconvert --to script "Jupyter_UI.ipynb"